# ETL-23 · Tabela fato

- **Etapa:** ETL-23, tabela fato (`fato_pessoa_sinistro`)
- **Responsável:** Davi Mello
- **Issue:** ETL-23
- **Entradas:** a base limpa (saídas das ETL-13 a ETL-17 em `data/intermediarios/`) e as 8 dimensões em `data/intermediarios/dims/` (ETL-19 a ETL-22)
- **Saída:** `data/intermediarios/fato.parquet`

Monta a fato no grão de **uma pessoa em um sinistro**: cada grupo de colunas descritivas vira a chave `sk_` da sua dimensão, com duas chaves para a `dim_tempo` (data do sinistro e data do óbito). As colunas seguem o DDL da ETL-18 (`sql/ddl/etl_star_schema.sql`).

## 1. Setup

In [1]:
from pathlib import Path
import pandas as pd

# Raiz do repositório: sobe de pasta em pasta até achar a pasta .git
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".git").exists())

INTERM = RAIZ / "data" / "intermediarios"      # .parquet (o .gitignore já ignora)
DIMS   = INTERM / "dims"                       # as 8 dimensões
SAIDA  = INTERM / "fato.parquet"               # resultado desta etapa

COLUNAS_TRANSPORTE = ['tipo_de_vitima', 'modo_transporte_vitima', 'tipo_veiculo_vitima']
COLUNAS_PERFIL = ['sexo', 'faixa_etaria_demografica', 'faixa_etaria_legal', 'nacionalidade', 'grau_de_instrucao']
COLUNAS_VIA = ['tipo_via', 'local_via']

# Cada grupo de colunas vem do mesmo arquivo que gerou a sua dimensão, para os valores baterem.
# Na execução final (ETL-26) todos apontam para o mesmo arquivo: a saída da última etapa de limpeza.
FONTES = {
    'ids (ETL-13)':        (INTERM / '02_chaves_duplicatas.parquet', ['id_sinistro', 'id_veiculo']),
    'datas (ETL-15)':      (INTERM / '04_datas.parquet', ['data_sinistro', 'data_obito', 'dias_ate_obito']),
    'localidade (ETL-14)': (INTERM / '03_texto.parquet', ['cod_ibge']),
    'categorias (ETL-16)': (INTERM / '05_nulos_categorias.parquet',
                            ['gravidade_lesao', 'local_obito', 'idade', 'flag_obito',
                             *COLUNAS_TRANSPORTE, *COLUNAS_PERFIL, *COLUNAS_VIA]),
    'profissão (ETL-17)':  (INTERM / '06_limpo.parquet', ['profissao']),
}

## 2. Carregar a entrada

### 2.1 Base limpa (juntando os grupos pelo `id_pessoa`)

In [2]:
def carregar_base(fontes: dict) -> pd.DataFrame:
    """Lê cada grupo de colunas do seu arquivo e junta tudo pelo id_pessoa, sem perder nem duplicar linhas."""
    base = None
    for nome, (arquivo, colunas) in fontes.items():
        parte = pd.read_parquet(arquivo, columns=['id_pessoa', *colunas])
        parte['id_pessoa'] = pd.to_numeric(parte['id_pessoa'], errors='raise').astype('int64')
        assert parte['id_pessoa'].is_unique, f'{nome}: id_pessoa repetido'
        if base is None:
            base = parte
        else:
            n = len(base)
            base = base.merge(parte, on='id_pessoa', how='inner', validate='one_to_one')
            assert len(base) == n == len(parte), f'{nome}: pessoas diferentes das outras fontes'
        print(f'{nome:20s} | {arquivo.name:30s} | {len(parte)} linhas')
    return base

base = carregar_base(FONTES)
print('\nbase:', base.shape)

ids (ETL-13)         | 02_chaves_duplicatas.parquet   | 806977 linhas


datas (ETL-15)       | 04_datas.parquet               | 806977 linhas
localidade (ETL-14)  | 03_texto.parquet               | 806977 linhas


categorias (ETL-16)  | 05_nulos_categorias.parquet    | 806977 linhas
profissão (ETL-17)   | 06_limpo.parquet               | 806977 linhas

base: (806977, 22)


### 2.2 As 8 dimensões

In [3]:
NOMES_DIMS = ['dim_tempo', 'dim_gravidade', 'dim_perfil_vitima', 'dim_transporte',
              'dim_localidade', 'dim_via', 'dim_profissao', 'dim_local_obito']
dims = {nome: pd.read_parquet(DIMS / f'{nome}.parquet') for nome in NOMES_DIMS}

pd.DataFrame({'linhas': {n: len(d) for n, d in dims.items()},
              'membros -1 e -2': {n: int(d.iloc[:, 0].isin([-1, -2]).sum()) for n, d in dims.items()}})

,linhas,membros -1 e -2
dim_tempo,1121,2
dim_gravidade,6,2
dim_perfil_vitima,1500,2
dim_transporte,25,2
dim_localidade,647,2
dim_via,10,2
dim_profissao,561,2
dim_local_obito,5,2


## 3. Diagnóstico ANTES

In [4]:
def diagnostico(df: pd.DataFrame) -> pd.DataFrame:
    """Tabela-resumo usada antes e depois: tipo, % de nulos e nº de valores distintos por coluna."""
    return pd.DataFrame({
        'dtype': df.dtypes.astype(str),
        'pct_nulo': (df.isna().mean() * 100).round(2),
        'n_distintos': df.nunique(),
    })

antes = diagnostico(base)
antes

,dtype,pct_nulo,n_distintos
id_pessoa,int64,0.00,806977
id_sinistro,int64,0.00,402695
id_veiculo,Int64,21.04,554294
data_sinistro,datetime64[us],0.00,1095
data_obito,datetime64[us],98.04,1109
dias_ate_obito,Int64,98.04,31
cod_ibge,int64,0.00,645
gravidade_lesao,str,0.00,5
local_obito,str,0.00,5
idade,Int64,1.04,103


In [5]:
# números da base limpa que a fato tem que repetir
BASE_LINHAS    = len(base)
BASE_SINISTROS = base['id_sinistro'].nunique()
BASE_OBITOS    = int(base['flag_obito'].sum())
print('pessoas:', BASE_LINHAS, '| sinistros:', BASE_SINISTROS, '| óbitos:', BASE_OBITOS)

pessoas: 806977 | sinistros: 402695 | óbitos: 15793


## 4. Trocar as colunas pelas chaves

Cada função devolve a chave de cada pessoa na dimensão e **para com erro** se alguma pessoa não achar chave. Assim nenhum valor inesperado vira -1 em silêncio. O valor ausente só vai para -1 (NÃO INFORMADO) ou -2 (NÃO SE APLICA) pelas regras de cada dimensão.

### 4.1 Funções de busca

In [6]:
def conferir_chave(sk: pd.Series, nome: str) -> pd.Series:
    """Para com erro se alguma pessoa ficou sem chave; senão devolve a chave como inteiro."""
    sem_chave = sk.isna()
    assert not sem_chave.any(), f'{nome}: {int(sem_chave.sum())} pessoas sem chave'
    return sk.astype('int64')


def sk_por_combinacao(df: pd.DataFrame, dim: pd.DataFrame, colunas: list, sk: str) -> pd.Series:
    """Dimensões de combinações (perfil, transporte, via): busca pela combinação das colunas.
    As combinações só de NÃO INFORMADO ou só de NÃO SE APLICA acham os membros -1 e -2 pelo rótulo."""
    chaves = df[colunas].fillna('NÃO INFORMADO').replace('NAO DISPONIVEL', 'NÃO INFORMADO')
    m = chaves.merge(dim[[*colunas, sk]], on=colunas, how='left', validate='many_to_one')
    return conferir_chave(pd.Series(m[sk].to_numpy(), index=df.index), sk)


def sk_tempo(datas: pd.Series, sem_data: int) -> pd.Series:
    """dim_tempo: a chave é a data no formato AAAAMMDD; quem não tem data recebe sem_data."""
    if not pd.api.types.is_datetime64_any_dtype(datas):
        datas = pd.to_datetime(datas, format='%d/%m/%Y', errors='raise')
    sk = datas.dt.year * 10000 + datas.dt.month * 100 + datas.dt.day
    return sk.fillna(sem_data).astype('int64')


NIVEIS = {'ILESO': 0, 'LEVE': 1, 'GRAVE': 2, 'FATAL': 3}
def sk_gravidade(gravidade: pd.Series, dim: pd.DataFrame) -> pd.Series:
    """dim_gravidade: busca pelo nome; NAO DISPONIVEL e vazio vão para -1."""
    mapa = dim.set_index('gravidade_lesao')['sk_gravidade']
    nome = gravidade.fillna('NÃO INFORMADO').replace('NAO DISPONIVEL', 'NÃO INFORMADO')
    return conferir_chave(nome.map(mapa), 'sk_gravidade')


def sk_localidade(cod_ibge: pd.Series, dim: pd.DataFrame) -> pd.Series:
    """dim_localidade: busca pelo código IBGE."""
    reais = dim[dim['sk_localidade'] > 0]
    mapa = reais.set_index(reais['cod_ibge'].astype('int64'))['sk_localidade']
    return conferir_chave(pd.to_numeric(cod_ibge, errors='raise').map(mapa), 'sk_localidade')


NAO_INFORMADO_PROFISSAO = ['NAO INFORMADA', 'NAO INFORMADO', 'NÃO INFORMADO']
def sk_profissao(profissao: pd.Series, dim: pd.DataFrame) -> pd.Series:
    """dim_profissao: vazio e as grafias de "não informado" vão para -1 (regra da ETL-22)."""
    reais = dim[dim['sk_profissao'] > 0]
    mapa = reais.set_index('profissao')['sk_profissao']
    chave = profissao.mask(profissao.isin(NAO_INFORMADO_PROFISSAO))
    sk = chave.map(mapa)
    sk[chave.isna()] = -1
    return conferir_chave(sk, 'sk_profissao')


def sk_local_obito(local: pd.Series, dim: pd.DataFrame) -> pd.Series:
    """dim_local_obito: busca pelo nome (NÃO INFORMADO e NÃO SE APLICA já vêm da ETL-16)."""
    mapa = dim.set_index('local_obito')['sk_local_obito']
    return conferir_chave(local.map(mapa), 'sk_local_obito')

### 4.2 Montar a fato

In [7]:
def montar_fato(base: pd.DataFrame, dims: dict) -> pd.DataFrame:
    """Uma linha por pessoa: 3 identificadores da fonte, 9 chaves e 4 medidas, na ordem do DDL."""
    fato = pd.DataFrame(index=base.index)

    # dimensões degeneradas (identificadores da fonte)
    fato['id_pessoa'] = base['id_pessoa'].astype('int64')
    fato['id_sinistro'] = base['id_sinistro'].astype('int64')
    fato['id_veiculo'] = base['id_veiculo'].astype('Int64')      # nulo permitido

    # chaves das dimensões
    fato['sk_data_sinistro'] = sk_tempo(base['data_sinistro'], sem_data=-1)
    fato['sk_data_obito'] = sk_tempo(base['data_obito'], sem_data=-2)   # -2: quem não morreu
    fato['sk_gravidade'] = sk_gravidade(base['gravidade_lesao'], dims['dim_gravidade'])
    fato['sk_perfil_vitima'] = sk_por_combinacao(base, dims['dim_perfil_vitima'], COLUNAS_PERFIL, 'sk_perfil_vitima')
    fato['sk_transporte'] = sk_por_combinacao(base, dims['dim_transporte'], COLUNAS_TRANSPORTE, 'sk_transporte')
    fato['sk_localidade'] = sk_localidade(base['cod_ibge'], dims['dim_localidade'])
    fato['sk_via'] = sk_por_combinacao(base, dims['dim_via'], COLUNAS_VIA, 'sk_via')
    fato['sk_profissao'] = sk_profissao(base['profissao'], dims['dim_profissao'])
    fato['sk_local_obito'] = sk_local_obito(base['local_obito'], dims['dim_local_obito'])

    # medidas
    fato['idade'] = base['idade'].astype('Int16')
    fato['dias_ate_obito'] = base['dias_ate_obito'].astype('Int16')
    fato['flag_obito'] = base['flag_obito'].astype('int16')
    fato['qtd_pessoas'] = pd.Series(1, index=base.index, dtype='int16')
    return fato.reset_index(drop=True)

fato = montar_fato(base, dims)
print(fato.shape)
fato.head()

(806977, 16)


,id_pessoa,id_sinistro,id_veiculo,sk_data_sinistro,sk_data_obito,sk_gravidade,sk_perfil_vitima,sk_transporte,sk_localidade,sk_via,sk_profissao,sk_local_obito,idade,dias_ate_obito,flag_obito,qtd_pessoas
0,1979797,2240338,2874666,20210101,-2,1,184,14,585,8,228,-2,31,<NA>,0,1
1,1979798,2240338,2874666,20210101,-2,1,839,1,585,8,385,-2,34,<NA>,0,1
2,1284984,1447800,<NA>,20210101,-2,1,444,22,563,8,210,-2,65,<NA>,0,1
3,563279,1783410,2648403,20210101,-2,3,760,1,563,8,80,-2,26,<NA>,0,1
4,687322,1798856,<NA>,20210101,-2,2,1311,19,532,8,52,-2,66,<NA>,0,1


## 5. Conferências

Seis conferências: os totais batem com a base limpa, toda chave existe na sua dimensão, as regras (`CHECK`) do DDL valem, as medidas batem com as datas, a fato devolve os valores originais quando ligada de volta às dimensões e o total por ano bate com a fonte.

In [8]:
# 1. totais: linhas, pessoas, sinistros e óbitos iguais aos da base limpa
totais = pd.DataFrame({
    'base limpa': [BASE_LINHAS, BASE_LINHAS, BASE_SINISTROS, BASE_OBITOS],
    'fato': [len(fato), fato['id_pessoa'].nunique(), fato['id_sinistro'].nunique(), int(fato['flag_obito'].sum())],
}, index=['linhas', 'pessoas (id_pessoa)', 'sinistros (id_sinistro)', 'óbitos (flag_obito)'])
totais['confere'] = totais['base limpa'] == totais['fato']
assert totais['confere'].all()
totais

,base limpa,fato,confere
linhas,806977,806977,True
pessoas (id_pessoa),806977,806977,True
sinistros (id_sinistro),402695,402695,True
óbitos (flag_obito),15793,15793,True


In [9]:
# 2. integridade referencial: toda chave da fato existe na sua dimensão
FKS = {
    'sk_data_sinistro': ('dim_tempo', 'sk_tempo'),
    'sk_data_obito': ('dim_tempo', 'sk_tempo'),
    'sk_gravidade': ('dim_gravidade', 'sk_gravidade'),
    'sk_perfil_vitima': ('dim_perfil_vitima', 'sk_perfil_vitima'),
    'sk_transporte': ('dim_transporte', 'sk_transporte'),
    'sk_localidade': ('dim_localidade', 'sk_localidade'),
    'sk_via': ('dim_via', 'sk_via'),
    'sk_profissao': ('dim_profissao', 'sk_profissao'),
    'sk_local_obito': ('dim_local_obito', 'sk_local_obito'),
}
for fk, (dim, pk) in FKS.items():
    assert fato[fk].notna().all() and fato[fk].isin(dims[dim][pk]).all(), f'{fk}: chave fora da {dim}'
print('as 9 chaves estrangeiras existem nas dimensões e nenhuma é nula')

as 9 chaves estrangeiras existem nas dimensões e nenhuma é nula


In [10]:
# 3. regras do DDL (as mesmas CHECK que o PostgreSQL vai cobrar na carga)
sem_obito = fato['flag_obito'] == 0
regras = {
    'id_pessoa único (PK)': fato['id_pessoa'].is_unique,
    'ids cabem em INTEGER': bool(fato[['id_pessoa', 'id_sinistro']].max().max() < 2**31
                                 and fato['id_veiculo'].max() < 2**31),
    'idade entre 0 e 120 ou nula': bool(fato['idade'].dropna().between(0, 120).all()),
    'dias_ate_obito entre 0 e 30 ou nulo': bool(fato['dias_ate_obito'].dropna().between(0, 30).all()),
    'flag_obito é 0 ou 1': bool(fato['flag_obito'].isin([0, 1]).all()),
    'qtd_pessoas sempre 1': bool((fato['qtd_pessoas'] == 1).all()),
    'sem óbito: data -2, local -2 e dias nulo': bool((
        (fato.loc[sem_obito, 'sk_data_obito'] == -2) & (fato.loc[sem_obito, 'sk_local_obito'] == -2)
        & fato.loc[sem_obito, 'dias_ate_obito'].isna()).all()),
    'com óbito: data diferente de -2': bool((fato.loc[~sem_obito, 'sk_data_obito'] != -2).all()),
}
regras = pd.Series(regras, name='ok')
assert regras.all()
regras.to_frame()

,ok
id_pessoa único (PK),True
ids cabem em INTEGER,True
idade entre 0 e 120 ou nula,True
dias_ate_obito entre 0 e 30 ou nulo,True
flag_obito é 0 ou 1,True
qtd_pessoas sempre 1,True
"sem óbito: data -2, local -2 e dias nulo",True
com óbito: data diferente de -2,True


In [11]:
# 4. as medidas batem com as datas: dias_ate_obito = data do óbito - data do sinistro
datas_sin = pd.to_datetime(fato['sk_data_sinistro'].astype(str), format='%Y%m%d')
obitos = fato['sk_data_obito'] > 0
datas_obi = pd.to_datetime(fato.loc[obitos, 'sk_data_obito'].astype(str), format='%Y%m%d')
diferenca = (datas_obi - datas_sin[obitos]).dt.days
assert (diferenca == fato.loc[obitos, 'dias_ate_obito'].astype('int64')).all()
print('dias_ate_obito confere com as duas datas em', int(obitos.sum()), 'óbitos')

dias_ate_obito confere com as duas datas em 15793 óbitos


In [12]:
# 5. ida e volta: ligando a fato às dimensões, cada pessoa recupera os valores da base limpa
def normalizado(serie: pd.Series) -> pd.Series:
    """Mesma regra de ausente das dimensões: vazio e NAO DISPONIVEL viram NÃO INFORMADO."""
    return serie.fillna('NÃO INFORMADO').replace('NAO DISPONIVEL', 'NÃO INFORMADO')

COMPARAR = [   # (chave na fato, dimensão, colunas da dimensão = colunas da base)
    ('sk_gravidade', 'dim_gravidade', ['gravidade_lesao']),
    ('sk_perfil_vitima', 'dim_perfil_vitima', COLUNAS_PERFIL),
    ('sk_transporte', 'dim_transporte', COLUNAS_TRANSPORTE),
    ('sk_via', 'dim_via', COLUNAS_VIA),
    ('sk_local_obito', 'dim_local_obito', ['local_obito']),
]
divergencias = {}
for fk, nome, colunas in COMPARAR:
    pk = FKS[fk][1]
    volta = fato[[fk]].merge(dims[nome], left_on=fk, right_on=pk, how='left')
    for c in colunas:
        divergencias[c] = int((volta[c].to_numpy() != normalizado(base[c]).to_numpy()).sum())

# localidade (pelo código IBGE) e profissão (com a regra de "não informado" da ETL-22)
volta = fato[['sk_localidade']].merge(dims['dim_localidade'], on='sk_localidade', how='left')
divergencias['cod_ibge'] = int((volta['cod_ibge'].astype('int64').to_numpy() != base['cod_ibge'].to_numpy()).sum())
volta = fato[['sk_profissao']].merge(dims['dim_profissao'], on='sk_profissao', how='left')
prof = base['profissao'].mask(base['profissao'].isin(NAO_INFORMADO_PROFISSAO)).fillna('NÃO INFORMADO')
divergencias['profissao'] = int((volta['profissao'].to_numpy() != prof.to_numpy()).sum())

# datas
divergencias['data_sinistro'] = int((datas_sin.to_numpy() != base['data_sinistro'].to_numpy()).sum())
divergencias['data_obito'] = int((datas_obi.to_numpy() != base.loc[obitos, 'data_obito'].to_numpy()).sum())

divergencias = pd.Series(divergencias, name='pessoas com valor diferente')
assert (divergencias == 0).all()
divergencias.to_frame()

,pessoas com valor diferente
gravidade_lesao,0
sexo,0
faixa_etaria_demografica,0
faixa_etaria_legal,0
nacionalidade,0
grau_de_instrucao,0
tipo_de_vitima,0
modo_transporte_vitima,0
tipo_veiculo_vitima,0
tipo_via,0


In [13]:
# 6. pessoas e óbitos por ano do sinistro, pela dim_tempo (fonte: 260.886 / 266.128 / 279.963)
por_ano = (fato.merge(dims['dim_tempo'][['sk_tempo', 'ano']], left_on='sk_data_sinistro', right_on='sk_tempo')
               .groupby('ano')[['qtd_pessoas', 'flag_obito']].sum()
               .rename(columns={'qtd_pessoas': 'pessoas', 'flag_obito': 'óbitos'}))
assert por_ano['pessoas'].tolist() == [260886, 266128, 279963]
por_ano

,pessoas,óbitos
ano,,
2021,260886,4858
2022,266128,5445
2023,279963,5490


## 6. Diagnóstico DEPOIS

In [14]:
depois = diagnostico(fato)
depois

,dtype,pct_nulo,n_distintos
id_pessoa,int64,0.00,806977
id_sinistro,int64,0.00,402695
id_veiculo,Int64,21.04,554294
sk_data_sinistro,int64,0.00,1095
sk_data_obito,int64,0.00,1110
sk_gravidade,int64,0.00,5
sk_perfil_vitima,int64,0.00,1499
sk_transporte,int64,0.00,24
sk_localidade,int64,0.00,645
sk_via,int64,0.00,9


In [15]:
# uso dos membros especiais em cada chave (pessoas)
pd.DataFrame({
    '-1 (NÃO INFORMADO)': {fk: int((fato[fk] == -1).sum()) for fk in FKS},
    '-2 (NÃO SE APLICA)': {fk: int((fato[fk] == -2).sum()) for fk in FKS},
})

,-1 (NÃO INFORMADO),-2 (NÃO SE APLICA)
sk_data_sinistro,0,0
sk_data_obito,0,791184
sk_gravidade,70314,0
sk_perfil_vitima,3395,0
sk_transporte,98947,0
sk_localidade,0,0
sk_via,663,0
sk_profissao,86834,0
sk_local_obito,105,791184


## 7. Salvar a saída

In [16]:
fato.to_parquet(SAIDA, index=False)
print('salvo em:', SAIDA.relative_to(RAIZ), '| shape:', fato.shape)

salvo em: data/intermediarios/fato.parquet | shape: (806977, 16)


## 8. Consulta de teste no esquema estrela (só consulta)

Uma pergunta respondida só com a fato e as dimensões, como será feito no PostgreSQL: letalidade por gravidade registrada e por tipo de dia do sinistro.

In [17]:
tempo = dims['dim_tempo'].set_index('sk_tempo')
dia = tempo.loc[fato['sk_data_sinistro'], ['fim_de_semana', 'feriado']].reset_index(drop=True)
tipo_dia = pd.Series('DIA ÚTIL', index=fato.index)
tipo_dia[dia['fim_de_semana'].to_numpy(dtype=bool)] = 'FIM DE SEMANA'
tipo_dia[dia['feriado'].to_numpy(dtype=bool)] = 'FERIADO'

consulta = (fato.assign(tipo_dia=tipo_dia)
                .groupby('tipo_dia')[['qtd_pessoas', 'flag_obito']].sum()
                .rename(columns={'qtd_pessoas': 'pessoas', 'flag_obito': 'óbitos'}))
consulta['letalidade (%)'] = (consulta['óbitos'] / consulta['pessoas'] * 100).round(2)
consulta.reindex(['DIA ÚTIL', 'FIM DE SEMANA', 'FERIADO'])

,pessoas,óbitos,letalidade (%)
tipo_dia,,,
DIA ÚTIL,554750,9243,1.67
FIM DE SEMANA,234889,6039,2.57
FERIADO,17338,511,2.95


## 9. Resumo

**O que foi feito:** montagem da `fato_pessoa_sinistro`, com uma linha por pessoa envolvida em um sinistro. As 23 colunas aproveitadas da fonte viraram 3 identificadores (`id_pessoa`, `id_sinistro` e `id_veiculo`), 9 chaves para as 8 dimensões (duas para a `dim_tempo`: data do sinistro e data do óbito) e 4 medidas (`idade`, `dias_ate_obito`, `flag_obito` e `qtd_pessoas`). A saída foi salva em `data/intermediarios/fato.parquet`.

**Resultado:** 806.977 linhas e 16 colunas, na ordem do DDL da ETL-18.

**O que foi conferido:**
- **Totais:** 806.977 pessoas, 402.695 sinistros e 15.793 óbitos, iguais aos da base limpa. Por ano, 260.886, 266.128 e 279.963 pessoas, iguais aos da fonte.
- **Chaves:** as 9 chaves existem nas dimensões e nenhuma é nula. Nenhuma pessoa ficou sem chave: as funções de busca param com erro em vez de mandar um valor inesperado para o -1.
- **Regras do DDL:** valem todas as `CHECK` da tabela, inclusive a coerência do óbito. Quem não morreu tem data e local do óbito -2 e `dias_ate_obito` nulo.
- **Medidas:** `dias_ate_obito` é igual à diferença entre as duas datas em todos os óbitos.
- **Ida e volta:** ligando a fato às dimensões, cada pessoa recupera exatamente os valores da base limpa (gravidade, perfil, transporte, via, município, profissão, local e datas).

**Membros especiais:** o -2 aparece só onde o dado não se aplica: 791.184 pessoas sem óbito, na data e no local do óbito. O -1 aparece onde o dado faltou na fonte, por exemplo nas 70.314 pessoas sem gravidade e nas 86.834 sem profissão.

**Entradas:** cada grupo de colunas vem do mesmo arquivo que gerou a sua dimensão (ETL-13 a ETL-17), juntado pelo `id_pessoa`. Na execução final (ETL-26), todos apontam para o mesmo arquivo.

**Próximo passo:** a ETL-24 carrega as 8 dimensões e esta fato no PostgreSQL, nessa ordem, por causa das chaves estrangeiras.